# Pandas 기초

Pandas는 표 형태(테이블) 데이터를 다루는 Python의 대표 라이브러리입니다.
실무 ML 프로젝트에서 시간의 대부분은 모델링이 아니라 **데이터를 불러오고, 확인하고, 정제하는 일**에 쓰이는데, 그 대부분을 Pandas로 합니다.

## 학습 목표
- Series와 DataFrame의 구조 이해
- 데이터 확인 (head, info, describe, value_counts)
- 데이터 선택 — `loc` vs `iloc` 구분
- 조건 필터링과 열 추가/수정
- 그룹화와 집계 (groupby, pivot_table)
- 결측치 처리 전략
- 데이터 합치기 (concat, merge)
- 파일 저장/로딩

## 이 노트북 사용법
각 섹션은 **개념 설명 → 예제 코드 → ✏️ 직접 해보기 → 정답** 순서로 구성되어 있습니다.

> ✏️ **직접 해보기** 문제가 나오면 반드시 빈 셀에 직접 코드를 작성해 보세요.
> 정답은 그 아래에 있지만, 먼저 스스로 풀어본 뒤 확인해야 실력이 늡니다.

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas 버전: {pd.__version__}")

## 1. Series와 DataFrame

Pandas의 핵심 자료구조는 딱 두 개입니다:

| 구조 | 차원 | 비유 |
|------|------|------|
| `Series` | 1차원 | 엑셀의 **한 열** (값 + 인덱스) |
| `DataFrame` | 2차원 | 엑셀의 **시트 전체** (여러 Series를 묶은 표) |

NumPy 배열과의 결정적 차이는 **인덱스(라벨)** 가 붙어 있다는 것입니다.

In [ ]:
# Series: 값 + 인덱스
s = pd.Series([1, 3, 5, 7, 9])
print(s)
print(f"\n값(NumPy 배열): {s.values}")
print(f"인덱스: {s.index}")

In [ ]:
# 인덱스를 직접 지정한 Series — 딕셔너리처럼 라벨로 접근 가능
prices = pd.Series([1500, 3000, 2500], index=['아메리카노', '라떼', '카푸치노'])
print(prices)
print(f"\n라떼 가격: {prices['라떼']}")

# 딕셔너리로도 바로 만들 수 있음
s2 = pd.Series({'a': 100, 'b': 200, 'c': 300})
print(f"\n딕셔너리로 생성:\n{s2}")

In [ ]:
# DataFrame: 딕셔너리(열 이름 → 값 리스트)로 생성하는 것이 기본
df = pd.DataFrame({
    '이름': ['홍길동', '이순신', '강감찬'],
    '나이': [25, 30, 28],
    '점수': [90, 85, 95]
})
print(df)

# DataFrame의 핵심 속성
print(f"\nshape:   {df.shape}")      # (행 수, 열 수)
print(f"columns: {list(df.columns)}")
print(f"dtypes:\n{df.dtypes}")       # 열마다 타입이 다를 수 있음!

In [ ]:
# DataFrame에서 열 하나를 꺼내면 → Series
col = df['점수']
print(f"타입: {type(col).__name__}")
print(col)

# Series에는 통계 메서드가 바로 붙음
print(f"\n평균: {col.mean()}, 최대: {col.max()}")

### ✏️ 직접 해보기 1 — Series와 DataFrame 만들기

1. 과일 가격 Series를 만드세요. 인덱스는 `['사과', '바나나', '포도']`, 값은 `[3000, 4000, 6000]`. 그리고 '바나나'의 가격을 라벨로 접근해 출력하세요.
2. 위 Series에서 가격이 3500원보다 비싼 과일만 필터링하세요. (힌트: NumPy 불리언 인덱싱과 동일)
3. 다음 데이터를 DataFrame으로 만들고 `shape`과 `dtypes`를 출력하세요.
   - 도시: 서울, 부산, 대구 / 인구(만명): 950, 340, 240 / 광역시여부: True, True, True

In [ ]:
# TODO: 1번 — 과일 가격 Series


In [ ]:
# TODO: 2번 — 3500원 초과 과일


In [ ]:
# TODO: 3번 — 도시 DataFrame


In [ ]:
# ===== 정답 =====
fruits = pd.Series([3000, 4000, 6000], index=['사과', '바나나', '포도'])
print(f"1번 바나나 가격: {fruits['바나나']}\n")

print(f"2번:\n{fruits[fruits > 3500]}\n")

cities = pd.DataFrame({
    '도시': ['서울', '부산', '대구'],
    '인구': [950, 340, 240],
    '광역시여부': [True, True, True]
})
print(f"3번 shape: {cities.shape}")
print(cities.dtypes)

## 2. 데이터 확인

새 데이터를 받으면 **가장 먼저** 하는 일이 데이터 파악입니다. 아래 5개는 습관처럼 실행하세요.

| 메서드 | 확인하는 것 |
|--------|-------------|
| `df.head()` / `df.tail()` | 실제 데이터 생김새 |
| `df.info()` | 행 수, 열별 타입, **결측치 유무** |
| `df.describe()` | 숫자 열의 기초 통계 (이상치 탐지에 유용) |
| `df['열'].value_counts()` | 범주형 열의 값별 개수 |
| `df['열'].unique()` / `nunique()` | 고유값 목록 / 개수 |

In [ ]:
# 이 노트북 전체에서 사용할 샘플 데이터 (학생 10명의 성적표)
np.random.seed(42)
df = pd.DataFrame({
    '이름': [f'학생_{i}' for i in range(1, 11)],
    '국어': np.random.randint(50, 100, 10),
    '영어': np.random.randint(50, 100, 10),
    '수학': np.random.randint(50, 100, 10),
    '등급': np.random.choice(['A', 'B', 'C'], 10)
})
df

In [ ]:
# info: 행/열 개수, 타입, 결측치 여부를 한눈에
df.info()

In [ ]:
# describe: 숫자 열의 기초 통계
# min/max가 상식적인 범위인지 확인 → 이상치(outlier) 탐지의 첫걸음
df.describe()

In [ ]:
print(f"상위 3행:\n{df.head(3)}\n")
print(f"등급별 개수:\n{df['등급'].value_counts()}\n")
print(f"등급 고유값: {df['등급'].unique()}, 종류 수: {df['등급'].nunique()}")

In [ ]:
# 정렬: sort_values
print("수학 점수 상위 3명:")
print(df.sort_values('수학', ascending=False).head(3))

# nlargest: 정렬+head를 한 번에
print("\nnlargest로 같은 결과:")
print(df.nlargest(3, '수학'))

### ✏️ 직접 해보기 2 — 데이터 확인

위에서 만든 `df`(학생 성적표)를 사용하세요.

1. 하위 2개 행을 출력하세요.
2. 국어 점수가 낮은 순으로 정렬해서 하위 3명을 출력하세요.
3. 등급이 몇 종류인지, 각 등급에 몇 명씩 있는지 출력하세요.
4. `describe()` 결과에서 영어 점수의 평균과 최댓값을 읽어보고, 코드로도 직접 구해서 일치하는지 확인하세요.

In [ ]:
# TODO: 1번 — 하위 2개 행


In [ ]:
# TODO: 2번 — 국어 하위 3명


In [ ]:
# TODO: 3번 — 등급 종류 수와 등급별 인원


In [ ]:
# TODO: 4번 — 영어 평균/최댓값


In [ ]:
# ===== 정답 =====
print(f"1번:\n{df.tail(2)}\n")
print(f"2번:\n{df.nsmallest(3, '국어')}\n")   # 또는 df.sort_values('국어').head(3)
print(f"3번: {df['등급'].nunique()}종류\n{df['등급'].value_counts()}\n")
print(f"4번: 영어 평균={df['영어'].mean()}, 최댓값={df['영어'].max()}")

## 3. 데이터 선택 — `loc` vs `iloc`

Pandas에서 가장 헷갈리는 부분입니다. 확실히 구분해 두세요.

| | `loc` | `iloc` |
|---|-------|--------|
| 기준 | **라벨**(이름) | **위치**(정수 순서) |
| 슬라이스 끝 | **포함** ⚠️ | 미포함 (Python 방식) |
| 예 | `df.loc[0:2, '이름':'국어']` | `df.iloc[0:2, 0:2]` |

> ⚠️ `df.loc[0:2]`는 라벨 0~2로 **3개 행**, `df.iloc[0:2]`는 위치 0~1로 **2개 행**입니다.

In [ ]:
# 열 선택
print("열 하나 (Series):")
print(df['이름'].head(3))

print("\n여러 열 (DataFrame) — 리스트를 넣으므로 대괄호 2겹:")
print(df[['이름', '국어']].head(3))

In [ ]:
# loc: 라벨 기반
print("인덱스 라벨 0인 행:")
print(df.loc[0])

print("\n라벨 0~2행, '이름'~'국어' 열 (양쪽 끝 포함!):")
print(df.loc[0:2, '이름':'국어'])

In [ ]:
# iloc: 위치 기반
print("0~2번째 행, 0~1번째 열 (끝 미포함):")
print(df.iloc[0:3, 0:2])

print("\n마지막 행:")
print(df.iloc[-1])

In [ ]:
# loc vs iloc 차이를 직접 확인 — 같은 0:2인데 행 수가 다름!
print(f"loc[0:2]  → {len(df.loc[0:2])}행 (라벨 0,1,2 포함)")
print(f"iloc[0:2] → {len(df.iloc[0:2])}행 (위치 0,1만)")

### ✏️ 직접 해보기 3 — 데이터 선택

1. '이름'과 '수학' 열만 뽑아 상위 5행을 출력하세요.
2. `iloc`로 3~5번째 행(위치 3, 4, 5)의 모든 열을 출력하세요.
3. `loc`로 라벨 2~4행의 '영어'와 '수학' 열을 출력하세요.
4. `iloc`로 마지막 2개 행의 마지막 2개 열을 출력하세요. (힌트: 음수 인덱스)

In [ ]:
# TODO: 1번 — 이름, 수학 열 상위 5행


In [ ]:
# TODO: 2번 — iloc로 위치 3~5행


In [ ]:
# TODO: 3번 — loc로 라벨 2~4행의 영어, 수학


In [ ]:
# TODO: 4번 — 마지막 2행, 마지막 2열


In [ ]:
# ===== 정답 =====
print(f"1번:\n{df[['이름', '수학']].head()}\n")
print(f"2번:\n{df.iloc[3:6]}\n")
print(f"3번:\n{df.loc[2:4, ['영어', '수학']]}\n")
print(f"4번:\n{df.iloc[-2:, -2:]}")

## 4. 필터링

조건에 맞는 **행**을 골라내는 작업입니다. NumPy 불리언 인덱싱과 원리가 같습니다.

- 복수 조건은 `&`(and), `|`(or), `~`(not) — 각 조건을 **반드시 괄호**로 감싸기
- `isin([...])`: 여러 값 중 하나에 해당하는지
- `between(a, b)`: a 이상 b 이하
- `str.contains('문자열')`: 문자열 포함 여부

In [ ]:
# 단일 조건
print("국어 80점 이상:")
print(df[df['국어'] >= 80])

In [ ]:
# 복수 조건 — 괄호 필수!
print("국어 80점 이상 AND 영어 80점 이상:")
print(df[(df['국어'] >= 80) & (df['영어'] >= 80)])

print("\n국어 90점 이상 OR 수학 90점 이상:")
print(df[(df['국어'] >= 90) | (df['수학'] >= 90)])

In [ ]:
# isin, between, str.contains
print("등급이 A 또는 B:")
print(df[df['등급'].isin(['A', 'B'])])

print("\n수학이 60~80점 사이:")
print(df[df['수학'].between(60, 80)])

print("\n이름에 '1'이 들어가는 학생:")
print(df[df['이름'].str.contains('1')])

### ✏️ 직접 해보기 4 — 필터링

1. 영어 점수가 70점 미만인 학생을 출력하세요.
2. 등급이 C이면서 수학이 70점 이상인 학생을 출력하세요.
3. 국어, 영어, 수학 중 **하나라도** 90점 이상인 학생을 출력하세요.
4. 등급이 A가 **아닌** 학생 수를 구하세요. (힌트: `~` 또는 `!=`)

In [ ]:
# TODO: 1번 — 영어 70점 미만


In [ ]:
# TODO: 2번 — 등급 C이면서 수학 70점 이상


In [ ]:
# TODO: 3번 — 한 과목이라도 90점 이상


In [ ]:
# TODO: 4번 — 등급 A가 아닌 학생 수


In [ ]:
# ===== 정답 =====
print(f"1번:\n{df[df['영어'] < 70]}\n")
print(f"2번:\n{df[(df['등급'] == 'C') & (df['수학'] >= 70)]}\n")
print(f"3번:\n{df[(df['국어'] >= 90) | (df['영어'] >= 90) | (df['수학'] >= 90)]}\n")
print(f"4번: {len(df[df['등급'] != 'A'])}명")   # 또는 (~(df['등급'] == 'A')).sum()

## 5. 열 추가 / 수정 / 삭제

- 새 열 추가: `df['새열'] = 값` — 벡터 연산으로 계산한 결과를 바로 넣을 수 있음
- 조건부 값: `np.where(조건, 참값, 거짓값)` 이 `apply(lambda ...)`보다 훨씬 빠름
- 여러 구간으로 나누기: `pd.cut`
- 삭제: `df.drop(columns=[...])`, 이름 변경: `df.rename(columns={...})`

In [ ]:
# 평균 열 추가 — 열끼리 벡터 연산
df['평균'] = (df['국어'] + df['영어'] + df['수학']) / 3
df['평균'] = df['평균'].round(1)
df.head()

In [ ]:
# 조건부 열: np.where (권장 — 빠름)
df['합격여부'] = np.where(df['평균'] >= 70, '합격', '불합격')

# apply + lambda (복잡한 로직일 때만)
df['평가'] = df['평균'].apply(lambda x: '우수' if x >= 85 else ('보통' if x >= 70 else '노력'))
df.head()

In [ ]:
# pd.cut: 숫자를 구간(범주)으로 나누기 — 나이대, 점수대 등에 자주 사용
df['점수대'] = pd.cut(df['평균'], bins=[0, 60, 70, 80, 100],
                    labels=['F', 'C', 'B', 'A'])
df[['이름', '평균', '점수대']].head()

In [ ]:
# 열 삭제와 이름 변경 — 원본을 바꾸려면 다시 할당 (또는 inplace=True)
df = df.drop(columns=['평가', '점수대'])
df = df.rename(columns={'합격여부': '결과'})
df.head()

### ✏️ 직접 해보기 5 — 열 추가/수정

1. 세 과목 중 최고 점수를 담은 '최고점수' 열을 추가하세요. (힌트: `df[['국어','영어','수학']].max(axis=1)`)
2. 최고점수와 최저점수의 차이를 담은 '편차' 열을 추가하세요.
3. `np.where`로 수학이 국어보다 높으면 '이과형', 아니면 '문과형'인 '유형' 열을 추가하세요.
4. '편차' 열을 삭제하세요.

In [ ]:
# TODO: 1번 — 최고점수 열


In [ ]:
# TODO: 2번 — 편차 열


In [ ]:
# TODO: 3번 — 유형 열 (np.where)


In [ ]:
# TODO: 4번 — 편차 열 삭제


In [ ]:
# ===== 정답 =====
subjects = df[['국어', '영어', '수학']]
df['최고점수'] = subjects.max(axis=1)
df['편차'] = subjects.max(axis=1) - subjects.min(axis=1)
df['유형'] = np.where(df['수학'] > df['국어'], '이과형', '문과형')
print(df.head())

df = df.drop(columns=['편차'])
print(f"\n4번 삭제 후 열: {list(df.columns)}")

## 6. 그룹화와 집계

`groupby`는 **"나누고(split) → 계산하고(apply) → 합친다(combine)"** 패턴입니다.
SQL의 `GROUP BY`, 엑셀의 피벗 테이블과 같은 개념입니다.

```
df.groupby('기준열')['집계할열'].집계함수()
```

In [ ]:
# 등급별 평균 점수
print("등급별 평균:")
print(df.groupby('등급')[['국어', '영어', '수학']].mean().round(1))

print("\n등급별 학생 수:")
print(df.groupby('등급').size())

In [ ]:
# agg: 여러 집계 함수를 한 번에
print("등급별 평균 점수의 통계:")
print(df.groupby('등급')['평균'].agg(['mean', 'min', 'max', 'count']).round(1))

# 열마다 다른 집계 함수 적용
print("\n열마다 다른 집계:")
print(df.groupby('등급').agg(
    국어평균=('국어', 'mean'),
    수학최고=('수학', 'max'),
    인원=('이름', 'count')
).round(1))

In [ ]:
# 여러 기준으로 그룹화
print("등급 x 결과별 인원:")
print(df.groupby(['등급', '결과']).size())

# pivot_table: 그룹화 결과를 표 형태로
print("\npivot_table 버전:")
print(pd.pivot_table(df, index='등급', columns='결과', values='이름',
                     aggfunc='count', fill_value=0))

### ✏️ 직접 해보기 6 — 그룹화

1. 등급별 수학 점수의 최댓값을 구하세요.
2. '결과'(합격/불합격)별로 평균 점수의 평균을 구하세요.
3. 등급별로 국어는 평균, 영어는 최솟값, 인원수를 한 번에 집계하세요. (힌트: 이름 있는 agg)
4. `pivot_table`로 등급(행) × 유형(열)별 평균 점수의 평균을 구하세요.

In [ ]:
# TODO: 1번 — 등급별 수학 최댓값


In [ ]:
# TODO: 2번 — 결과별 평균


In [ ]:
# TODO: 3번 — 열마다 다른 집계


In [ ]:
# TODO: 4번 — pivot_table


In [ ]:
# ===== 정답 =====
print(f"1번:\n{df.groupby('등급')['수학'].max()}\n")
print(f"2번:\n{df.groupby('결과')['평균'].mean().round(1)}\n")
print("3번:")
print(df.groupby('등급').agg(
    국어평균=('국어', 'mean'),
    영어최저=('영어', 'min'),
    인원=('이름', 'count')
).round(1))
print("\n4번:")
print(pd.pivot_table(df, index='등급', columns='유형', values='평균', aggfunc='mean').round(1))

## 7. 결측치 처리

실제 데이터에는 빈 값(NaN)이 흔합니다. ML 모델 대부분은 결측치가 있으면 학습이 안 되므로 반드시 처리해야 합니다.

| 방법 | 코드 | 언제 쓰나 |
|------|------|-----------|
| 확인 | `df.isnull().sum()` | 항상 제일 먼저 |
| 행 제거 | `df.dropna()` | 결측치가 적고 데이터가 충분할 때 |
| 값 채우기 | `df.fillna(값)` | 평균/중앙값/0 등으로 대체 |
| 앞/뒤 값으로 | `df.ffill()` / `df.bfill()` | 시계열 데이터 |

> 💡 무엇으로 채울지는 데이터의 의미에 따라 다릅니다. 무조건 평균이 정답은 아닙니다.

In [ ]:
# 결측치가 있는 데이터
df_null = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5],
    'B': [np.nan, 2, 3, 4, np.nan],
    'C': [1, 2, 3, 4, 5]
})
print("원본:")
print(df_null)

print(f"\n열별 결측치 개수:\n{df_null.isnull().sum()}")
print(f"\n결측치 비율(%):\n{(df_null.isnull().mean() * 100).round(1)}")

In [ ]:
# 전략 1: 평균으로 채우기 (열별 평균이 각각 적용됨)
print("평균으로 채우기:")
print(df_null.fillna(df_null.mean()))

# 전략 2: 특정 값으로 채우기
print("\n0으로 채우기:")
print(df_null.fillna(0))

In [ ]:
# 전략 3: 결측치 포함 행 제거
print(f"dropna 후: {len(df_null.dropna())}행 (원본 {len(df_null)}행)")
print(df_null.dropna())

# 전략 4: 시계열식 — 앞의 값으로 채우기
print("\nffill (앞의 값으로):")
print(df_null.ffill())

### ✏️ 직접 해보기 7 — 결측치

다음 데이터를 사용하세요.

```python
sensor = pd.DataFrame({
    '온도': [22.1, np.nan, 23.5, np.nan, 24.0, 22.8],
    '습도': [45, 47, np.nan, 50, 52, np.nan],
    '기기': ['A', 'A', 'B', 'B', 'C', 'C']
})
```

1. 열별 결측치 개수를 확인하세요.
2. 온도는 **중앙값**(median)으로, 습도는 **평균**으로 채운 DataFrame을 만드세요.
3. 원본에서 결측치가 있는 행을 모두 제거하면 몇 행이 남는지 확인하세요.
4. (도전) 온도를 `ffill`로 채우면 1번 행(인덱스 1)의 온도가 얼마가 될지 예상해 보고 확인하세요.

In [ ]:
sensor = pd.DataFrame({
    '온도': [22.1, np.nan, 23.5, np.nan, 24.0, 22.8],
    '습도': [45, 47, np.nan, 50, 52, np.nan],
    '기기': ['A', 'A', 'B', 'B', 'C', 'C']
})

# TODO: 1번 — 결측치 개수


In [ ]:
# TODO: 2번 — 온도는 중앙값, 습도는 평균으로


In [ ]:
# TODO: 3번 — dropna 후 행 수


In [ ]:
# TODO: 4번 — ffill 결과 예상 후 확인


In [ ]:
# ===== 정답 =====
print(f"1번:\n{sensor.isnull().sum()}\n")

filled = sensor.fillna({'온도': sensor['온도'].median(), '습도': sensor['습도'].mean()})
print(f"2번:\n{filled}\n")

print(f"3번: {len(sensor.dropna())}행\n")

print(f"4번: ffill은 바로 앞 값(22.1)을 가져옴\n{sensor['온도'].ffill()}")

## 8. 데이터 합치기 — concat과 merge

여러 개의 표를 하나로 합치는 작업입니다. 실무에서 데이터는 항상 여러 파일/테이블로 쪼개져 옵니다.

| 함수 | 용도 | 비유 |
|------|------|------|
| `pd.concat([df1, df2])` | 같은 구조의 표를 **이어붙이기** | 종이 두 장을 위아래로 붙임 |
| `pd.merge(df1, df2, on='키')` | 공통 열(키)을 기준으로 **연결** | SQL의 JOIN, 엑셀의 VLOOKUP |

In [ ]:
# concat: 위아래로 이어붙이기 (예: 1월 데이터 + 2월 데이터)
jan = pd.DataFrame({'이름': ['홍길동', '이순신'], '매출': [100, 200]})
feb = pd.DataFrame({'이름': ['강감찬', '김유신'], '매출': [150, 300]})

total = pd.concat([jan, feb], ignore_index=True)  # 인덱스를 새로 매김
print(total)

In [ ]:
# merge: 공통 열을 기준으로 연결
students = pd.DataFrame({'학번': [1, 2, 3], '이름': ['홍길동', '이순신', '강감찬']})
grades = pd.DataFrame({'학번': [1, 2, 4], '성적': [90, 85, 70]})

# inner: 양쪽에 다 있는 학번만 (기본값)
print("inner join:")
print(pd.merge(students, grades, on='학번'))

# left: 왼쪽(students)은 전부 유지, 없으면 NaN
print("\nleft join:")
print(pd.merge(students, grades, on='학번', how='left'))

### ✏️ 직접 해보기 8 — 데이터 합치기

```python
products = pd.DataFrame({'상품코드': ['P1', 'P2', 'P3'], '상품명': ['노트북', '마우스', '키보드']})
sales = pd.DataFrame({'상품코드': ['P1', 'P1', 'P2', 'P4'], '수량': [1, 2, 5, 3]})
```

1. 두 표를 `inner` merge하세요. 몇 행이 나오는지, 왜 P3와 P4가 빠졌는지 생각해 보세요.
2. `left` merge(기준: sales)로 합치면 P4의 상품명이 어떻게 되는지 확인하세요.
3. merge한 결과에서 상품명별 총 판매 수량을 구하세요. (merge + groupby 조합)

In [ ]:
products = pd.DataFrame({'상품코드': ['P1', 'P2', 'P3'], '상품명': ['노트북', '마우스', '키보드']})
sales = pd.DataFrame({'상품코드': ['P1', 'P1', 'P2', 'P4'], '수량': [1, 2, 5, 3]})

# TODO: 1번 — inner merge


In [ ]:
# TODO: 2번 — left merge (sales 기준)


In [ ]:
# TODO: 3번 — 상품명별 총 판매 수량


In [ ]:
# ===== 정답 =====
inner = pd.merge(sales, products, on='상품코드')
print(f"1번 ({len(inner)}행 — P3는 판매기록 없음, P4는 상품정보 없음):\n{inner}\n")

left = pd.merge(sales, products, on='상품코드', how='left')
print(f"2번 (P4의 상품명은 NaN):\n{left}\n")

print(f"3번:\n{inner.groupby('상품명')['수량'].sum()}")

## 9. 데이터 저장 / 로딩

| 형식 | 저장 | 로딩 | 특징 |
|------|------|------|------|
| CSV | `df.to_csv()` | `pd.read_csv()` | 가장 보편적. 한글은 `encoding='utf-8-sig'` |
| JSON | `df.to_json()` | `pd.read_json()` | API 연동에 자주 사용 |
| Excel | `df.to_excel()` | `pd.read_excel()` | `openpyxl` 패키지 필요 |
| Parquet | `df.to_parquet()` | `pd.read_parquet()` | 대용량에 빠르고 작음 (실무 표준) |

In [ ]:
# CSV 저장 — index=False가 중요 (안 쓰면 의미 없는 인덱스 열이 같이 저장됨)
df.to_csv('students.csv', index=False, encoding='utf-8-sig')
print("students.csv 저장 완료")

# 다시 읽기
df_loaded = pd.read_csv('students.csv')
print(df_loaded.head(3))

In [ ]:
# JSON 저장
df.to_json('students.json', force_ascii=False, orient='records', indent=2)
print("students.json 저장 완료")

df_json = pd.read_json('students.json')
print(df_json.head(3))

## 10. 종합 연습 문제

지금까지 배운 내용을 모두 활용하는 문제입니다. 힌트 없이 스스로 풀어보세요!

먼저 아래 셀을 실행해 직원 데이터를 만드세요.

**문제 1.** 부서별 평균 연봉을 높은 순으로 정렬해 출력하세요.

**문제 2.** 연봉이 상위 3위 안에 드는 직원의 이름과 연봉을 출력하세요.

**문제 3.** '개발' 부서이면서 경력 5년 이상인 직원을 필터링하세요.

**문제 4.** 경력 구간별('주니어': 0~3년, '미들': 4~7년, '시니어': 8년~) '직급' 열을 추가하고, 직급별 평균 연봉을 구하세요. (힌트: `pd.cut`)

**문제 5.** 나이 열의 결측치를 **부서별 평균 나이**로 채우세요. (힌트: `groupby` + `transform`)

**문제 6.** 부서(행) × 직급(열)별 인원수 pivot_table을 만드세요. 빈 칸은 0으로.

In [ ]:
# 종합 문제용 데이터
np.random.seed(7)
emp = pd.DataFrame({
    '이름': [f'직원{i}' for i in range(1, 13)],
    '부서': np.random.choice(['개발', '기획', '영업'], 12),
    '연봉': np.random.randint(3000, 8000, 12),
    '경력': np.random.randint(0, 12, 12),
    '나이': [28, 32, np.nan, 41, 29, np.nan, 35, 27, 38, np.nan, 31, 44]
})
emp

In [ ]:
# TODO: 문제 1 — 부서별 평균 연봉 (내림차순)


In [ ]:
# TODO: 문제 2 — 연봉 상위 3명


In [ ]:
# TODO: 문제 3 — 개발 부서 & 경력 5년 이상


In [ ]:
# TODO: 문제 4 — 직급 열 추가 + 직급별 평균 연봉


In [ ]:
# TODO: 문제 5 — 나이 결측치를 부서별 평균으로


In [ ]:
# TODO: 문제 6 — 부서 x 직급 인원수 pivot_table


In [ ]:
# ===== 정답 1 =====
print(emp.groupby('부서')['연봉'].mean().sort_values(ascending=False).round(0))

In [ ]:
# ===== 정답 2 =====
print(emp.nlargest(3, '연봉')[['이름', '연봉']])

In [ ]:
# ===== 정답 3 =====
print(emp[(emp['부서'] == '개발') & (emp['경력'] >= 5)])

In [ ]:
# ===== 정답 4 =====
emp['직급'] = pd.cut(emp['경력'], bins=[-1, 3, 7, 100],
                   labels=['주니어', '미들', '시니어'])
print(emp[['이름', '경력', '직급']])
print(f"\n직급별 평균 연봉:\n{emp.groupby('직급', observed=True)['연봉'].mean().round(0)}")

In [ ]:
# ===== 정답 5 =====
# transform: 그룹별 계산 결과를 원본과 같은 길이로 돌려줌 → fillna에 바로 사용 가능
emp['나이'] = emp['나이'].fillna(emp.groupby('부서')['나이'].transform('mean').round(0))
print(emp[['이름', '부서', '나이']])

In [ ]:
# ===== 정답 6 =====
print(pd.pivot_table(emp, index='부서', columns='직급', values='이름',
                     aggfunc='count', fill_value=0, observed=True))

## 정리

| 주제 | 핵심 내용 |
|------|-----------|
| 자료구조 | Series(1차원) / DataFrame(2차원), 열 하나 꺼내면 Series |
| 데이터 확인 | `head`, `info`, `describe`, `value_counts` — 새 데이터 받으면 무조건 실행 |
| 선택 | `loc`은 라벨(끝 포함), `iloc`은 위치(끝 미포함) |
| 필터링 | `df[조건]`, 복수 조건은 `&`/`\|` + 괄호, `isin`, `between` |
| 열 추가 | 벡터 연산, `np.where`, `pd.cut` |
| 그룹화 | `groupby` + `agg`, `pivot_table`, `transform` |
| 결측치 | `isnull().sum()` → `fillna` / `dropna` (전략은 데이터에 따라) |
| 합치기 | `concat`(이어붙이기) vs `merge`(키로 연결, inner/left) |

다음 단계: **Matplotlib & Seaborn** (`03-visualization/matplotlib-seaborn.ipynb`)